#### Luis Guillermo Rivera Stephens
## Lab 3

In [62]:
from SparkInit import init_spark

spark = init_spark("Lab 3")

In [63]:
columns_types = [
    ("Order_ID", "long"),
    ("Company", "string"),
    ("City", "string"),
    ("Customer_Age", "int"),
    ("Order_Value", "float"),
    ("Delivery_Time_Min", "float"),
    ("Distance_Km", "float"),
    ("Items_Count", "float"),
    ("Product_Category", "string"),
    ("Payment_Method", "string"),
    ("Customer_Rating", "float"),
    ("Discount_Applied", "int"),
    ("Delivery_Partner_Rating", "float"),
]

In [64]:
from SchemaUtils import SchemaUtils

qcommerce_schema = SchemaUtils.create_schema(columns_types)
print(qcommerce_schema)

StructType([StructField('Order_ID', LongType(), True), StructField('Company', StringType(), True), StructField('City', StringType(), True), StructField('Customer_Age', IntegerType(), True), StructField('Order_Value', FloatType(), True), StructField('Delivery_Time_Min', FloatType(), True), StructField('Distance_Km', FloatType(), True), StructField('Items_Count', FloatType(), True), StructField('Product_Category', StringType(), True), StructField('Payment_Method', StringType(), True), StructField('Customer_Rating', FloatType(), True), StructField('Discount_Applied', IntegerType(), True), StructField('Delivery_Partner_Rating', FloatType(), True)])


In [65]:
pwd

'/opt/spark/work-dir'

In [66]:
ls data/qcommerce

quick_commerce_data_raw.csv*


In [67]:
df = spark \
                .read.option("header", "true") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/qcommerce/")
df.printSchema()
df.show()


root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: integer (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)

+--------+----------------+---------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|     City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+

In [68]:
df.count()

1000000

In [69]:
from pyspark.sql.functions import when, count, isnull

df.columns

['Order_ID',
 'Company',
 'City',
 'Customer_Age',
 'Order_Value',
 'Delivery_Time_Min',
 'Distance_Km',
 'Items_Count',
 'Product_Category',
 'Payment_Method',
 'Customer_Rating',
 'Discount_Applied',
 'Delivery_Partner_Rating']

In [70]:
df_null_count = df.select([count(when(isnull(c), c)).alias(c) for c in df.columns])

df_null_count.show()

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [71]:
from SparkUtils import SparkUtils

df_null_count_2 = SparkUtils.count_nulls(df)
df_null_count_2.show()

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [72]:
df_wo_nulls = df.dropna()
df_wo_nulls.count()


780992

In [73]:
#fillna recibe un diccionario con el nombre de la columna y el valor a reemplazar

default_values = {
    "City": "Unknown",
    "Items_Count": 0,
    "Customer_Rating": 0,
    "Delivery_Partner_Rating": 0,
}

df_filled = df.fillna(default_values)
df_filled.count()

SparkUtils.count_nulls(df_filled).show()

+--------+-------+----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company|City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|   0|           0|          0|                0|          0|          0|               0|             0|              0|               0|                      0|
+--------+-------+----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [74]:
from pyspark.sql.functions import lit, col

df_ = df_filled.withColumn("Code_Country", lit("IN"))
df_.show()

+--------+----------------+---------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|     City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|
+--------+----------------+---------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|    Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|               1|                    3.2|          IN|
| 1000002|Flipkart Minutes| Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|          

In [75]:
df_tax = df_.withColumn("Paid_Tax", col("Order_Value") * 0.15)
df_tax.show()

+--------+----------------+---------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+------------------+
|Order_ID|         Company|     City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|          Paid_Tax|
+--------+----------------+---------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+------------------+
| 1000001|Swiggy Instamart|    Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|               1|                    3.2|          IN|105.35062866210937|
| 1000002|Flipkart Minutes| Amritsar|          56|     1007.3|           19.644|      12

In [76]:
df = df_tax.withColumn("Delivery_SLA", when(col("Delivery_Time_Min") <= 15, "FAST")  \
    .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON_TIME")  \
    .when(col("Delivery_Time_Min") > 20, "LATE"))

late_orders = df.filter(col("Delivery_SLA") == "LATE").orderBy(col("Delivery_Time_Min"), ascending=False)
late_orders.select("Order_ID","Delivery_Time_Min" ,"Delivery_SLA").show()



+--------+-----------------+------------+
|Order_ID|Delivery_Time_Min|Delivery_SLA|
+--------+-----------------+------------+
| 1529779|             40.0|        LATE|
| 1807126|             40.0|        LATE|
| 1165764|           39.994|        LATE|
| 1610720|           39.994|        LATE|
| 1729503|           39.994|        LATE|
| 1951122|           39.988|        LATE|
| 1975896|           39.988|        LATE|
| 1059671|           39.982|        LATE|
| 1594835|           39.982|        LATE|
| 1162804|           39.982|        LATE|
| 1826295|           39.982|        LATE|
| 1961544|           39.976|        LATE|
| 1360875|           39.964|        LATE|
| 1555058|           39.958|        LATE|
| 1709566|           39.958|        LATE|
| 1727081|           39.958|        LATE|
| 1894031|           39.958|        LATE|
| 1664387|           39.952|        LATE|
| 1308175|           39.946|        LATE|
| 1416405|           39.946|        LATE|
+--------+-----------------+------

In [77]:
eov_df = df.withColumn("Effective_Order_Value", col("Order_Value") - col("Discount_Applied"))
eov_df.select("Order_ID","Order_Value","Discount_Applied","Effective_Order_Value").show(5)

+--------+-----------+----------------+---------------------+
|Order_ID|Order_Value|Discount_Applied|Effective_Order_Value|
+--------+-----------+----------------+---------------------+
| 1000001|   702.3375|               1|    701.3375244140625|
| 1000002|     1007.3|               0|   1007.2999877929688|
| 1000003|    1211.66|               0|   1211.6600341796875|
| 1000004|  1179.0592|               1|   1178.0592041015625|
| 1000005|   586.0255|               0|    586.0255126953125|
+--------+-----------+----------------+---------------------+
only showing top 5 rows


In [78]:
price_bucket_df = eov_df.withColumn("Price_Bucket", when(col("Effective_Order_Value") < 200, "LOW")  \
    .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "MEDIUM")  \
    .when(col("Effective_Order_Value") > 500, "HIGH"))

price_bucket_df.select("Order_ID","Effective_Order_Value","Price_Bucket").show(5)


+--------+---------------------+------------+
|Order_ID|Effective_Order_Value|Price_Bucket|
+--------+---------------------+------------+
| 1000001|    701.3375244140625|        HIGH|
| 1000002|   1007.2999877929688|        HIGH|
| 1000003|   1211.6600341796875|        HIGH|
| 1000004|   1178.0592041015625|        HIGH|
| 1000005|    586.0255126953125|        HIGH|
+--------+---------------------+------------+
only showing top 5 rows


In [79]:
from pyspark.sql.functions import avg, count

aux_df = price_bucket_df.groupBy("Price_Bucket") \
    .agg(avg("Effective_Order_Value").alias("avg_effective_order_value"),  \
    count("*").alias("count_orders")) \
    .orderBy("avg_effective_order_value", ascending=False)

aux_df.show()

+------------+-------------------------+------------+
|Price_Bucket|avg_effective_order_value|count_orders|
+------------+-------------------------+------------+
|        HIGH|        846.8669016455311|      533001|
|      MEDIUM|        359.1362905200168|      288926|
|         LOW|        90.40103362272598|      178073|
+------------+-------------------------+------------+



In [80]:
cl = "Customer_Age" 
age_group_df = price_bucket_df.filter((col(cl).isNotNull()) & (col(cl) > 0) & (col(cl) < 100)) \
    .withColumn("Age_Group", when(col(cl) < 25, "YOUNG") \
        .when((col(cl) >= 25) & (col(cl) <= 44), "ADULT") \
        .when(col(cl) >= 45, "SENIOR"))


age_group_df.select("Customer_Age", "Age_Group").show(5)

result_df = age_group_df.groupBy("Age_Group", "Product_Category") \
    .agg(count("*").alias("orders"), \
    avg("Order_Value").alias("avg_order_value")) \
    .orderBy(col("Age_Group").asc(), col("orders").desc())


result_df.show()

+------------+---------+
|Customer_Age|Age_Group|
+------------+---------+
|          46|   SENIOR|
|          56|   SENIOR|
|          18|    YOUNG|
|          23|    YOUNG|
|          44|    ADULT|
+------------+---------+
only showing top 5 rows


+---------+-------------------+------+-----------------+
|Age_Group|   Product_Category|orders|  avg_order_value|
+---------+-------------------+------+-----------------+
|    ADULT|              Dairy| 68512|573.4268899414931|
|    ADULT|      Personal Care| 68331| 569.512671998805|
|    ADULT|          Groceries| 68291|571.5250993844182|
|    ADULT|          Household| 68110|572.9259149188181|
|    ADULT|             Snacks| 68100|570.3797974095505|
|    ADULT|          Beverages| 67979|572.0329877095578|
|    ADULT|Fruits & Vegetables| 67736|569.8593599596651|
|   SENIOR|          Groceries| 51030|572.2596391052539|
|   SENIOR|              Dairy| 51025| 573.781117268945|
|   SENIOR|             Snacks| 50924| 572.687851608881|
|   SENIOR|          Household| 50803| 571.172082385334|
|   SENIOR|          Beverages| 50746| 568.141013231256|
|   SENIOR|      Personal Care| 50707|571.1642560109325|
|   SENIOR|Fruits & Vegetables| 50642|573.7244422534909|
|    YOUNG|              Dairy|

In [81]:
spark.stop()